# Agreement of the grader with your labels

This notebook is for week 10, step 3.

1. **Label 50 outputs yourself** for the category of your grader: pass or fail. The labels go to `evals/grader_labels.jsonl`. Label before you run the grader, so that its decisions do not change yours.
2. **Run the grader** on the same 50 outputs. This sends 50 requests to the Claude API.
3. **Measure the agreement**: the true positive rate and the true negative rate, with `agreement_rates()` from `evals/grader.py`. The result goes to `evals/grader_agreement.json`.

Change the rubric in `evals/rubric.md` and run part 2 and part 3 again, until you accept both rates. The checks of week 10 read both files, so commit them.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from evals.grader import RUBRIC_PATH, agreement_rates, grade

LABELS = ROOT / "evals" / "grader_labels.jsonl"
AGREEMENT = ROOT / "evals" / "grader_agreement.json"

runs = sorted((ROOT / "evals" / "results").glob("testset-*.jsonl"))
assert runs, "No result file. Run `make testset` first."
rows = [json.loads(line) for run in runs for line in run.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"{len(rows)} outputs in {len(runs)} result files")

## Part 1: label 50 outputs

`show(n)` prints output number `n`. `label(n, passed, note)` saves your label. Take outputs that pass and outputs that fail: the checks need at least 10 of each. The failures of your notes of week 9 are a good start.

In [ ]:
def load_labels():
    return [json.loads(line) for line in LABELS.read_text(encoding="utf-8").splitlines() if line.strip()] if LABELS.exists() else []


def show(n):
    row = rows[n]
    print(f"[{n}] {row['id']}  {row['question']}")
    for tool in row["tools"]:
        print(f"  TOOL {tool['name']} {json.dumps(tool['input'], ensure_ascii=False)}")
    print("ANSWER:", row["answer"])
    print("DATASET IDS:", row["dataset_ids"], " CITATIONS:", [c.get("chunk_id") for c in row["citations"]])


def label(n, passed, note=""):
    row = rows[n]
    labels = [item for item in load_labels() if item["id"] != row["id"]]
    labels.append(dict(row, label="pass" if passed else "fail", label_note=note))
    LABELS.write_text("".join(json.dumps(item, ensure_ascii=False) + "\n" for item in labels), encoding="utf-8")
    counts = pd.Series([item["label"] for item in labels]).value_counts().to_dict()
    print(f"{len(labels)} labels: {counts}")


show(0)
# label(0, True)

## Part 2: run the grader on your labelled outputs

Each call of `grade()` is one request to Claude. With 50 outputs on Claude Haiku 5.5, a run costs some cents. Look at the request log for the real cost.

In [ ]:
labelled = load_labels()
verdicts = []
for item in labelled:
    verdict = grade(item)
    verdicts.append(verdict)
    print(f"{item['id']:6} label={item['label']:4} grader={'pass' if verdict.passed else 'fail'}")

## Part 3: the agreement rates

A positive is a failure. The true positive rate is the share of your failures that the grader also fails. The true negative rate is the share of your passes that the grader also passes. Read each disagreement: is the rubric unclear, or is your label wrong?

In [ ]:
human = [item["label"] == "pass" for item in labelled]
machine = [verdict.passed for verdict in verdicts]
tpr, tnr = agreement_rates(human, machine)
result = {
    "rubric": RUBRIC_PATH.read_text(encoding="utf-8"),
    "true_positive_rate": tpr,
    "true_negative_rate": tnr,
    "rows": [
        {"id": item["id"], "label": item["label"], "grader": "pass" if verdict.passed else "fail", "reasoning": verdict.reasoning}
        for item, verdict in zip(labelled, verdicts)
    ],
}
AGREEMENT.write_text(json.dumps(result, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
print(f"{len(labelled)} outputs: true positive rate {tpr:.2f}, true negative rate {tnr:.2f}")
table = pd.DataFrame(result["rows"])
table[table["label"] != table["grader"]]